In [2]:
import gensim
from gensim.models import Word2Vec, KeyedVectors

In [3]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

In [4]:
wv['king'].shape

(300,)

In [5]:
import pandas as pd
messages = pd.read_csv(r'Datasets/dataset.csv')
messages.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [6]:
import re
from nltk.corpus import stopwords

In [7]:
messages = messages[:2000]

In [8]:
from tqdm import tqdm
# It's just used to see the progress

In [9]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
corpus = []
for i in tqdm(range(0,len(messages))):
    review = re.sub('[^a-zA-Z]',' ',messages['text'][i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review if not word in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

100%|██████████| 2000/2000 [16:25<00:00,  2.03it/s] 


In [10]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [11]:
# simple_preprocess() # converts document into a list of lowercase tokens

In [12]:
words = []
for sent in corpus:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [13]:
words[:2]

[['viiiiiiagraaaa',
  'one',
  'want',
  'make',
  'scream',
  'prodigy',
  'scrawny',
  'crow',
  'define',
  'upgrade',
  'spongy',
  'balboa',
  'dither',
  'moiseyev',
  'schumann',
  'variegate',
  'ponce',
  'bernie',
  'cox',
  'angeles',
  'impassive',
  'circulate',
  'impend',
  'miscellany',
  'chalkboard',
  'whizzing',
  'pend',
  'armenian',
  'cutlet',
  'waring',
  'makeshift',
  'fletch',
  'dispel',
  'crest',
  'cadet',
  'dovetail',
  'rapprochement',
  'gerry',
  'bayreuth',
  'selectman',
  'wilmington',
  'tuttle',
  'alchemy',
  'itt',
  'bullyboy',
  'caan'],
 ['got',
  'ice',
  'thought',
  'look',
  'az',
  'original',
  'message',
  'ice',
  'operation',
  'mailto',
  'iceoperations',
  'intcx',
  'com',
  'sent',
  'friday',
  'october',
  'escapenumber',
  'escapenumber',
  'escapenumber',
  'escapenumber',
  'pm',
  'subject',
  'escapelong',
  'amended',
  'participant',
  'agreement',
  'dear',
  'participant',
  'receiving',
  'email',
  'identified',


### Train Word2Vec from scratch

In [14]:
model = gensim.models.Word2Vec(words,vector_size=100)

In [15]:
# To get all the vocabulary
model.wv.index_to_key[:5]

['escapenumber', 'escapelong', 'com', 'enron', 'http']

In [16]:
model.corpus_count

1999

In [17]:
model.epochs

5

In [18]:
model.wv.similar_by_word('good')
# These are the similar words found in given corpus.
# Not the googles pretrained corpus

[('look', 0.97928786277771),
 ('still', 0.9733483791351318),
 ('well', 0.97206711769104),
 ('back', 0.9705987572669983),
 ('really', 0.9683408141136169),
 ('going', 0.9665069580078125),
 ('keep', 0.957832932472229),
 ('come', 0.9572317600250244),
 ('thing', 0.9552549719810486),
 ('much', 0.9484792351722717)]

In [19]:
wv.similar_by_word('first')
# This is the googles model trained on 300 dimensions, and it's own WordNet

[('second', 0.7971885800361633),
 ('third', 0.6932076215744019),
 ('fourth', 0.6732367277145386),
 ('fifth', 0.6571477651596069),
 ('sixth', 0.6237859129905701),
 ('seventh', 0.591548502445221),
 ('thefirst', 0.5863957405090332),
 ('last', 0.5842218995094299),
 ('eighth', 0.5558103919029236),
 ('final', 0.5550165772438049)]

In [23]:
model.wv['first'].shape # this one was trained on only 100 dimensions

(100,)

In [24]:
# In normal Word2Vec
# each word will have 100 dimensions: each dimension explaining the similarity of it against the word.
# but in a given sentence there will be multiple words,leading to n * 100 vectors.
# Instead, for a given sentence we can take a average across all words for a given dimension.
# with this we can reduce the vector size of the sentence to 100
# this is called AvgWord2Vec
words[0] # This is one sentence, each of these words will have 100 dimension vectors.

['viiiiiiagraaaa',
 'one',
 'want',
 'make',
 'scream',
 'prodigy',
 'scrawny',
 'crow',
 'define',
 'upgrade',
 'spongy',
 'balboa',
 'dither',
 'moiseyev',
 'schumann',
 'variegate',
 'ponce',
 'bernie',
 'cox',
 'angeles',
 'impassive',
 'circulate',
 'impend',
 'miscellany',
 'chalkboard',
 'whizzing',
 'pend',
 'armenian',
 'cutlet',
 'waring',
 'makeshift',
 'fletch',
 'dispel',
 'crest',
 'cadet',
 'dovetail',
 'rapprochement',
 'gerry',
 'bayreuth',
 'selectman',
 'wilmington',
 'tuttle',
 'alchemy',
 'itt',
 'bullyboy',
 'caan']

In [25]:
import numpy as np

In [26]:
def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]
    if len(vectors) == 0:
        # If no sentence is present return the 0,0,0--0 vector
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [27]:
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 1999/1999 [00:17<00:00, 114.99it/s]


In [28]:
shapes = set(np.shape(x) for x in X)
print(shapes)

{(100,)}


In [29]:
len(X)

1999

In [30]:
X=np.array(X)

In [31]:
X[0].shape # Each sentence will have 100 dimensions

(100,)

In [32]:
X.shape

(1999, 100)

In [33]:
# Dependent Feature
y = pd.get_dummies(messages['label'])
y

,Ham,Spam
0,False,True
1,True,False
2,False,True
3,False,True
4,True,False
...,...,...
1995,False,True
1996,True,False
1997,True,False
1998,False,True


In [34]:
y = y.iloc[:,1].values #Taking either one column will do the job for considering o/p

In [35]:
X.shape

(1999, 100)

In [36]:
y.shape

(2000,)

In [37]:
y = messages[list(map(lambda x:len(x)>0,corpus))]
y = pd.get_dummies(y['label'])
y = y.iloc[:,0].values

In [38]:
y.shape

(1999,)

In [39]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

### Model Traning

In [40]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [41]:
model = classifier.fit(X_train,y_train)

In [42]:
y_pred = model.predict(X_test)

In [43]:
from sklearn.metrics import accuracy_score,classification_report
accuracy_score(y_test,y_pred)

0.885

In [44]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.85      0.91      0.88       187
        True       0.92      0.86      0.89       213

    accuracy                           0.89       400
   macro avg       0.88      0.89      0.88       400
weighted avg       0.89      0.89      0.89       400

